In [1]:
import pandas as pd

customers = pd.read_csv("../data/raw/customers.csv", parse_dates = ['signup_date'], dtype = {'birth_year': 'Int64'})
products = pd.read_csv("../data/raw/products.csv", parse_dates = ['launch_date', 'discontinued_date'])
orders = pd.read_csv("../data/raw/orders.csv", parse_dates = ['order_ts', 'promised_date', 'delivered_date'], dtype = {'session_id' : 'Int64'})
orders.insert(4, "order_date_local", orders["order_ts"].dt.tz_localize("UTC").dt.tz_convert("Europe/Vilnius").dt.tz_localize(None).dt.normalize())
order_items = pd.read_csv("../data/raw/order_items.csv", parse_dates = ['return_date'])
user_events = pd.read_csv("../data/raw/user_events.csv", parse_dates = ['event_ts'], dtype = {'customer_id': 'Int64', 'product_id': 'Int64'})
user_events.insert(6, "event_date_local", user_events["event_ts"].dt.tz_localize("UTC").dt.tz_convert("Europe/Vilnius").dt.tz_localize(None).dt.normalize())
marketing_spend = pd.read_csv("../data/raw/marketing_spend.csv", parse_dates = ['spend_date'], dtype = {'platform_conversions' : 'Int64'})

tables = {"customers": customers,
          "products": products,
          "orders": orders,
          "order_items": order_items,
          "user_events": user_events,
          "marketing_spend": marketing_spend
         }

## Profiling function

In [2]:
def profile(df, name, key):
    print(f"===== {name} =====")
    print(f"\n***** Shape *****\n")
    print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}\n")
    print(f"***** Dtypes *****\n")
    print(df.dtypes)
    print(f"\n***** Unique Key *****\n")
    print(f"How many duplicates in {', '.join(key)}? {df.duplicated(subset=key).sum()}")
    print(f"How many missing values in {', '.join(key)}? {df[key].isna().any(axis=1).sum()}")
    print(f"How many fully duplicated rows? {df.duplicated().sum()}")
    print(f"\n***** NULL values in columns *****\n")
    display(pd.DataFrame({'Missing values': df.isna().sum(), '% Missing': (df.isna().mean() * 100).round(2)}))
    print(f"\n***** MIN, MAX, PERCENTILES *****\n")
    display(df.select_dtypes(include="number").describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]).T)
    print(f"\n***** CATEGORICAL VARIABLES *****\n")
    
    text_cols = df.select_dtypes(exclude=["number", "datetime"]).columns
    for col in text_cols:
        n_unique = df[col].nunique()
        n_clean = df[col].str.strip().str.lower().nunique()
        print(f"\n----- {col}: {n_unique} unique -> {n_clean} after strip/lower -----\n")
        if n_unique <= 20:
            display(df[col].value_counts(dropna=False).to_frame("count"))
    print(f"\n***** DATE RANGE *****\n")
    display(df.select_dtypes(include="datetime").agg(["min", "max"]).T)

## CUSTOMERS

In [3]:
display(customers.head(10))

,customer_id,email_hash,signup_date,country,city,gender,birth_year,acquisition_source,marketing_opt_in
0,1,b57bf7094ed8,2023-12-28,LT,Test,NaN,<NA>,direct,0
1,2,8b7ec82f60c6,2024-01-01,LT,Šiauliai,Male,1999,organic_search,1
2,3,bbfae928f872,2024-01-02,LT,Klaipėda,Male,<NA>,direct,1
3,4,7548ae3fa95a,2024-01-02,LT,Kaunas,Female,1996,paid_social,1
4,5,528728837f93,2024-01-02,LT,Siauliai,Female,1991,paid_social,1
5,6,2ba1a69ceef4,2024-01-03,LT,Panevėžys,Female,1989,direct,0
6,7,7081b25e2e7e,2024-01-03,LT,Vilnius,Female,1979,email,0
7,8,d210317e87a9,2024-01-04,LT,Marijampolė,Male,1987,organic_search,1
8,9,c3c4cba41af4,2024-01-04,LT,Vilnius,Female,1993,direct,0
9,10,ffacef68e448,2024-01-05,EE,Narva,Male,1990,paid_search,1


In [4]:
profile(customers, "customers", ["customer_id"])

===== customers =====

***** Shape *****

Rows: 7483, Columns: 9

***** Dtypes *****

customer_id                    int64
email_hash                       str
signup_date           datetime64[us]
country                          str
city                             str
gender                           str
birth_year                     Int64
acquisition_source               str
marketing_opt_in               int64
dtype: object

***** Unique Key *****

How many duplicates in customer_id? 0
How many missing values in customer_id? 0
How many fully duplicated rows? 0

***** NULL values in columns *****



,Missing values,% Missing
customer_id,0,0.00
email_hash,0,0.00
signup_date,0,0.00
country,0,0.00
city,439,5.87
gender,483,6.45
birth_year,1009,13.48
acquisition_source,0,0.00
marketing_opt_in,0,0.00



***** MIN, MAX, PERCENTILES *****



,count,mean,std,min,1%,5%,25%,50%,75%,95%,99%,max
customer_id,7483.0,3742.0,2160.300365,1.0,75.82,375.1,1871.5,3742.0,5612.5,7108.9,7408.18,7483.0
birth_year,6474.0,1984.959376,11.947787,1900.0,1955.73,1964.0,1977.0,1986.0,1994.0,2002.0,2006.0,2026.0
marketing_opt_in,7483.0,0.57664,0.494124,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0



***** CATEGORICAL VARIABLES *****


----- email_hash: 7427 unique -> 7427 after strip/lower -----


----- country: 3 unique -> 3 after strip/lower -----



...

,count
gender,
F,3107
M,2527
Female,685
Male,593
NaN,483
X,69
Other,19



----- acquisition_source: 7 unique -> 7 after strip/lower -----



,count
acquisition_source,
organic_search,2104
paid_social,1433
paid_search,1281
direct,1052
referral,646
affiliate,563
email,404



***** DATE RANGE *****



,min,max
signup_date,2023-12-28,2026-09-30


## My questions
1. Are all 'signup_date' values within the data period stated in the data dictionary?
2. Which values in 'gender' and 'city' mean the same thing but are written differently?
4. Can one person have more than one account (for example, the same email_hash)?

## PRODUCTS

In [5]:
display(products.head(10))

,product_id,product_name,category,subcategory,brand,list_price,unit_cost,launch_date,discontinued_date
0,1,Lumea Reed Diffuser 100ml,Home & Kitchen,Decor,Lumea,19.99,84.00,2023-09-03,NaT
1,2,Strata Lightweight Running Jacket,Sports & Outdoors,Running,Strata,69.99,41.13,2023-09-03,NaT
2,3,Fjord Wear Men's Down Parka,Fashion,Outerwear,Fjord Wear,249.99,101.69,2023-09-04,NaT
3,4,Naturo Relaxing Massage Oil 200ml,Beauty & Care,Wellness,Naturo,14.99,8.56,2023-09-04,NaT
4,5,Pixora Indoor Security Camera 2K,Electronics,Smart Home,Pixora,49.99,35.03,2023-09-04,NaT
5,6,Silka Ionic Hair Dryer 2200W,Beauty & Care,Haircare,Silka,59.99,25.24,2023-09-04,NaT
6,7,Panela Stainless Steel Pot Set 5-Piece,Home & Kitchen,Cookware,Panela,139.99,72.36,2023-09-06,NaT
7,8,Whiskr Wet Cat Food 12x85g,Pets,Food,Whiskr,13.99,10.58,2023-09-06,NaT
8,9,Aktiva Football Size 5,Sports & Outdoors,Team Sports,Aktiva,24.99,15.40,2023-09-07,NaT
9,10,Dermia Men's Face Moisturizer 75ml,Beauty & Care,Skincare,Dermia,19.99,8.96,2023-09-07,NaT


In [6]:
profile(products, "products", ["product_id"])

===== products =====

***** Shape *****

Rows: 260, Columns: 9

***** Dtypes *****

product_id                    int64
product_name                    str
category                        str
subcategory                     str
brand                           str
list_price                  float64
unit_cost                   float64
launch_date          datetime64[us]
discontinued_date    datetime64[us]
dtype: object

***** Unique Key *****

How many duplicates in product_id? 0
How many missing values in product_id? 0
How many fully duplicated rows? 0

***** NULL values in columns *****



,Missing values,% Missing
product_id,0,0.00
product_name,0,0.00
category,0,0.00
subcategory,0,0.00
brand,0,0.00
list_price,0,0.00
unit_cost,0,0.00
launch_date,0,0.00
discontinued_date,232,89.23



***** MIN, MAX, PERCENTILES *****



,count,mean,std,min,1%,5%,25%,50%,75%,95%,99%,max
product_id,260.0,130.500000,75.199734,1.00,3.5900,13.950,65.7500,130.50,195.2500,247.050,257.4100,260.00
list_price,260.0,66.555385,105.835678,6.99,8.9900,11.990,22.9900,34.99,69.9900,200.490,620.4900,949.99
unit_cost,260.0,43.057769,93.898865,2.56,3.9272,5.546,10.5375,19.09,37.7525,134.808,571.7689,852.20



***** CATEGORICAL VARIABLES *****


----- product_name: 260 unique -> 260 after strip/lower -----


----- category: 10 unique -> 9 after strip/lower -----



,count
category,
Fashion,50
Home & Kitchen,43
Sports & Outdoors,35
Beauty & Care,35
Electronics,29
Toys & Kids,25
Pets,20
Garden,20
Home and Kitchen,2



----- subcategory: 37 unique -> 37 after strip/lower -----


----- brand: 48 unique -> 48 after strip/lower -----


***** DATE RANGE *****



,min,max
launch_date,2023-09-03,2026-07-15
discontinued_date,2024-11-18,2026-07-10


## My questions
1. Are there products with 'unit_cost' higher than 'list_price'? How many? Why?
2. Were any products sold before their 'launch_date' or after their 'discontinued_date'?
3. Are 'category', 'subcategory' and 'brand' values written consistently (no different spellings of the same value)?
4. Does every 'subcategory' belong to only one 'category'?

## ORDERS

In [7]:
orders.head(10)

,order_id,customer_id,session_id,order_ts,order_date_local,status,coupon_code,payment_method,delivery_method,shipping_fee,order_total,promised_date,delivered_date
0,1,2,15,2024-01-01 14:18:20,2024-01-01,completed,NaN,bank_link,parcel_locker,2.99,27.98,2024-01-03,2024-01-03
1,2,1,32,2024-01-02 07:12:30,2024-01-02,cancelled,NaN,card,pickup,0.00,0.01,2024-01-02,NaT
2,3,1,34,2024-01-02 07:40:02,2024-01-02,cancelled,NaN,card,pickup,0.00,0.01,2024-01-02,NaT
3,4,3,39,2024-01-02 10:45:31,2024-01-02,completed,NaN,card,courier,0.00,144.98,2024-01-03,2024-01-03
4,5,4,45,2024-01-02 14:52:26,2024-01-02,completed,WELCOME10,card,parcel_locker,2.99,24.57,2024-01-04,2024-01-04
5,6,5,48,2024-01-02 17:06:21,2024-01-02,completed,WELCOME10,bank_link,parcel_locker,0.00,181.76,2024-01-04,2024-01-04
6,7,6,65,2024-01-03 00:48:14,2024-01-03,cancelled,WELCOME10,bank_link,courier,4.99,27.48,2024-01-04,NaT
7,8,1,81,2024-01-03 12:05:47,2024-01-03,completed,NaN,card,pickup,0.00,0.01,2024-01-04,2024-01-03
8,9,7,83,2024-01-03 12:12:26,2024-01-03,completed,NaN,card,courier,0.00,52.98,2024-01-05,2024-01-05
9,10,9,139,2024-01-04 15:35:51,2024-01-04,completed,NaN,card,parcel_locker,2.99,38.96,2024-01-06,2024-01-06


In [11]:
profile(orders, "orders", ["order_id"])

===== orders =====

***** Shape *****

Rows: 10269, Columns: 13

***** Dtypes *****

order_id                     int64
customer_id                  int64
session_id                   Int64
order_ts            datetime64[us]
order_date_local    datetime64[us]
status                         str
coupon_code                    str
payment_method                 str
delivery_method                str
shipping_fee               float64
order_total                float64
promised_date       datetime64[us]
delivered_date      datetime64[us]
dtype: object

***** Unique Key *****

How many duplicates in order_id? 10
How many missing values in order_id? 0
How many fully duplicated rows? 10

***** NULL values in columns *****



,Missing values,% Missing
order_id,0,0.00
customer_id,0,0.00
session_id,584,5.69
order_ts,0,0.00
order_date_local,0,0.00
status,0,0.00
coupon_code,8411,81.91
payment_method,0,0.00
delivery_method,0,0.00
shipping_fee,0,0.00



***** MIN, MAX, PERCENTILES *****



,count,mean,std,min,1%,5%,25%,50%,75%,95%,99%,max
order_id,10269.0,5129.646996,2960.241604,1.0,103.68,514.4,2568.0,5125.0,7692.0,9745.6,10156.32,10259.0
customer_id,10269.0,3333.881196,2100.220309,1.0,61.0,308.4,1486.0,3161.0,5071.0,6894.6,7346.32,7482.0
session_id,9685.0,105026.223851,60290.575041,15.0,2141.88,10397.8,52473.0,107133.0,156546.0,199116.4,207423.64,209303.0
shipping_fee,10269.0,2.018672,2.284789,0.0,0.0,0.0,0.0,0.0,3.49,5.49,7.49,7.49
order_total,10269.0,72.027268,101.937539,0.01,11.98,15.18,27.98,46.98,78.27,193.282,612.4864,2018.97



***** CATEGORICAL VARIABLES *****


----- status: 4 unique -> 4 after strip/lower -----



...

,count
payment_method,
bank_link,4618
card,4530
paypal,581
pay_later,430
cash_on_delivery,110



----- delivery_method: 3 unique -> 3 after strip/lower -----



,count
delivery_method,
parcel_locker,6310
courier,3195
pickup,764



***** DATE RANGE *****



,min,max
order_ts,2024-01-01 14:18:20,2026-09-30 19:03:40
order_date_local,2024-01-01 00:00:00,2026-09-30 00:00:00
promised_date,2024-01-02 00:00:00,2026-10-03 00:00:00
delivered_date,2024-01-03 00:00:00,2026-09-30 00:00:00


## My questions
1. Are all 'order_ts', 'promised_date', 'delivered_date' values within the data period stated in the data dictionary? ('order_ts' is UTC time)
2. Are there any cancelled orders that have a 'delivered_date'?
3. Are there any orders whose local order date is later than 'promised_date' or 'delivered_date'?
4. Are there any orders that don't look like real customer orders ('order_total' close to 0)? Who placed them?

## ORDER_ITEMS

In [8]:
order_items.head(10)

,order_item_id,order_id,product_id,quantity,unit_price,discount_pct,is_returned,return_reason,return_date
0,1,1,31,1,24.99,0,0,NaN,NaT
1,2,2,111,1,0.01,0,0,NaN,NaT
2,3,3,136,1,0.01,0,0,NaN,NaT
3,4,4,7,1,119.99,0,0,NaN,NaT
4,5,4,85,1,24.99,0,0,NaN,NaT
5,6,5,78,1,11.99,10,0,NaN,NaT
6,7,5,132,1,11.99,10,0,NaN,NaT
7,8,6,7,1,119.99,10,0,NaN,NaT
8,9,6,41,1,19.99,10,1,changed_mind,2024-01-06
9,10,6,109,1,44.99,10,0,NaN,NaT


In [15]:
profile(order_items, "order_items", ["order_item_id"])

===== order_items =====

***** Shape *****

Rows: 17784, Columns: 9

***** Dtypes *****

order_item_id             int64
order_id                  int64
product_id                int64
quantity                  int64
unit_price              float64
discount_pct              int64
is_returned               int64
return_reason               str
return_date      datetime64[us]
dtype: object

***** Unique Key *****

How many duplicates in order_item_id? 0
How many missing values in order_item_id? 0
How many fully duplicated rows? 0

***** NULL values in columns *****



,Missing values,% Missing
order_item_id,0,0.00
order_id,0,0.00
product_id,0,0.00
quantity,0,0.00
unit_price,0,0.00
discount_pct,0,0.00
is_returned,0,0.00
return_reason,16312,91.72
return_date,16218,91.19



***** MIN, MAX, PERCENTILES *****



,count,mean,std,min,1%,5%,25%,50%,75%,95%,99%,max
order_item_id,17784.0,8892.500000,5133.942929,1.0,178.83,890.15,4446.75,8892.50,13338.25,16894.85,17606.17,17784.00
order_id,17784.0,5108.625337,2948.382302,1.0,99.83,522.15,2571.75,5110.00,7649.25,9740.00,10146.17,10259.00
product_id,17784.0,110.092780,62.689397,1.0,4.00,14.00,58.00,110.00,153.00,224.00,244.17,260.00
quantity,17784.0,1.086651,0.387445,1.0,1.00,1.00,1.00,1.00,1.00,2.00,3.00,15.00
unit_price,17784.0,39.933268,68.447364,0.0,8.49,10.49,14.99,27.99,39.99,99.99,259.99,1099.99
discount_pct,17784.0,4.712382,9.022094,0.0,0.00,0.00,0.00,0.00,10.00,25.00,40.00,50.00
is_returned,17784.0,0.088057,0.283385,0.0,0.00,0.00,0.00,0.00,0.00,1.00,1.00,1.00



***** CATEGORICAL VARIABLES *****


----- return_reason: 7 unique -> 7 after strip/lower -----



,count
return_reason,
NaN,16312
changed_mind,416
not_as_described,284
defective,249
size_too_small,198
size_too_large,172
arrived_late,84
wrong_item,69



***** DATE RANGE *****



,min,max
return_date,2024-01-06,2026-09-30


## My questions
1. Can the same 'product_id' appear more than once within one 'order_id'?
2. Are 'quantity', 'unit_price' and 'discount_pct' within realistic ranges?
3. When 'is_returned' = 1, are 'return_reason' and 'return_date' always filled in? When 'is_returned' = 0, are they always empty?

## Check: which lines have the lowest unit_price? (min was 0.00 in profile)

In [19]:
order_items.nsmallest(6, "unit_price")

,order_item_id,order_id,product_id,quantity,unit_price,discount_pct,is_returned,return_reason,return_date
2147,2148,1240,223,12,0.00,0,0,NaN,NaT
2148,2149,1240,259,10,0.00,0,0,NaN,NaT
1,2,2,111,1,0.01,0,0,NaN,NaT
2,3,3,136,1,0.01,0,0,NaN,NaT
12,13,8,121,1,0.01,0,0,NaN,NaT
360,361,208,117,1,6.49,10,0,NaN,NaT


## USER_EVENTS

In [9]:
user_events.head(10)

,event_id,session_id,visitor_id,customer_id,event_type,event_ts,event_date_local,device,traffic_source,product_id
0,1,1,7940527645,<NA>,view_item,2024-01-01 05:15:03,2024-01-01,mobile,organic_search,109
1,2,1,7940527645,<NA>,view_item,2024-01-01 05:18:46,2024-01-01,mobile,organic_search,111
2,3,1,7940527645,<NA>,view_item,2024-01-01 05:18:52,2024-01-01,mobile,organic_search,41
3,4,1,7940527645,<NA>,view_item,2024-01-01 05:19:04,2024-01-01,mobile,organic_search,7
4,5,1,7940527645,<NA>,view_item,2024-01-01 05:20:27,2024-01-01,mobile,organic_search,138
5,6,1,7940527645,<NA>,view_item,2024-01-01 05:21:04,2024-01-01,mobile,organic_search,131
6,7,2,2772890305,<NA>,view_item,2024-01-01 05:57:53,2024-01-01,mobile,affiliate,36
7,8,3,7286947083,<NA>,view_item,2024-01-01 06:09:33,2024-01-01,mobile,organic_search,84
8,9,4,7282267126,<NA>,view_item,2024-01-01 07:32:25,2024-01-01,desktop,organic_search,138
9,10,5,2227241223,<NA>,view_item,2024-01-01 07:56:38,2024-01-01,desktop,organic_search,17


In [20]:
profile(user_events, "user_events", ["event_id"])

===== user_events =====

***** Shape *****

Rows: 603272, Columns: 10

***** Dtypes *****

event_id                     int64
session_id                   int64
visitor_id                   int64
customer_id                  Int64
event_type                     str
event_ts            datetime64[us]
event_date_local    datetime64[us]
device                         str
traffic_source                 str
product_id                   Int64
dtype: object

***** Unique Key *****

How many duplicates in event_id? 0
How many missing values in event_id? 0
How many fully duplicated rows? 0

***** NULL values in columns *****



,Missing values,% Missing
event_id,0,0.00
session_id,0,0.00
visitor_id,0,0.00
customer_id,509422,84.44
event_type,0,0.00
event_ts,0,0.00
event_date_local,0,0.00
device,0,0.00
traffic_source,11944,1.98
product_id,34616,5.74



***** MIN, MAX, PERCENTILES *****



,count,mean,std,min,1%,5%,25%,50%,75%,95%,99%,max
event_id,603272.0,301636.5,174149.770135,1.0,6033.71,30164.55,150818.75,301636.5,452454.25,573108.45,597239.29,603272.0
session_id,603272.0,104772.981333,60393.950813,1.0,2107.0,10465.0,52485.0,105122.0,157272.25,198909.0,207250.0,209335.0
visitor_id,603272.0,5525060572.708505,2588376993.464117,1000097751.0,1092940140.0,1461158514.0,3308332201.0,5519829106.5,7785308333.5,9550829663.0,9913210932.0,9999973527.0
customer_id,93850.0,2594.888269,1901.668446,1.0,27.0,194.0,969.0,2242.5,3886.0,6247.0,7109.0,7483.0
product_id,568656.0,111.05032,61.835837,1.0,5.0,14.0,59.0,111.0,153.0,222.0,246.0,260.0



***** CATEGORICAL VARIABLES *****


----- event_type: 5 unique -> 5 after strip/lower -----



...

,count
device,
mobile,404744
desktop,168074
tablet,30454



----- traffic_source: 7 unique -> 7 after strip/lower -----



,count
traffic_source,
organic_search,153019
paid_social,117595
direct,102258
paid_search,82785
email,53008
referral,43266
affiliate,39397
NaN,11944



***** DATE RANGE *****



,min,max
event_ts,2024-01-01 05:15:03,2026-09-30 20:56:29
event_date_local,2024-01-01 00:00:00,2026-09-30 00:00:00


## My questions
1. Are all 'event_ts' (UTC) values within the data period stated in the data dictionary?
2. Can one 'session_id' have more than one 'visitor_id', 'device' or 'traffic_source'?
3. For which 'event_type' values is 'customer_id' filled in, and is it always filled in for purchase events?
4. Are there sessions or visitors with unusual activity (very many events, very little time between events) that could be bots?
5. Is 'product_id' filled in only for 'view_item' and 'add_to_cart' events?

## MARKETING_SPEND

In [10]:
marketing_spend.head(10)

,spend_date,channel,campaign_type,impressions,clicks,spend_eur,platform_conversions
0,2024-01-01,affiliate,commission,528,2,0.00,0
1,2024-01-01,email,newsletter_platform,7,6,1.58,0
2,2024-01-01,organic_search,seo_retainer,1076,29,14.52,<NA>
3,2024-01-01,paid_search,brand,7,2,0.27,0
4,2024-01-01,paid_search,non_brand,83,4,1.87,0
5,2024-01-01,paid_social,prospecting,223,2,0.57,0
6,2024-01-01,paid_social,retargeting,68,1,0.38,0
7,2024-01-02,affiliate,commission,606,2,0.00,0
8,2024-01-02,email,newsletter_platform,7,2,1.58,0
9,2024-01-02,organic_search,seo_retainer,911,30,14.52,<NA>


In [21]:
profile(marketing_spend, "marketing_spend", ["spend_date", "channel", "campaign_type"])

===== marketing_spend =====

***** Shape *****

Rows: 7026, Columns: 7

***** Dtypes *****

spend_date              datetime64[us]
channel                            str
campaign_type                      str
impressions                      int64
clicks                           int64
spend_eur                      float64
platform_conversions             Int64
dtype: object

***** Unique Key *****

How many duplicates in spend_date, channel, campaign_type? 4
How many missing values in spend_date, channel, campaign_type? 0
How many fully duplicated rows? 4

***** NULL values in columns *****



,Missing values,% Missing
spend_date,0,0.00
channel,0,0.00
campaign_type,0,0.00
impressions,0,0.00
clicks,0,0.00
spend_eur,0,0.00
platform_conversions,1004,14.29



***** MIN, MAX, PERCENTILES *****



,count,mean,std,min,1%,5%,25%,50%,75%,95%,99%,max
impressions,7026.0,3389.209935,4102.129039,4.0,11.0,19.0,162.25,1952.0,4794.0,11978.5,16854.5,50629.0
clicks,7026.0,48.843723,40.713068,0.0,2.0,7.0,20.0,35.0,66.0,134.0,173.75,454.0
spend_eur,7026.0,13.876937,14.924713,0.0,0.0,0.0,3.19,9.695,20.325,40.4475,65.0075,274.28
platform_conversions,6022.0,1.07456,1.620411,0.0,0.0,0.0,0.0,0.0,2.0,4.0,7.0,17.0



***** CATEGORICAL VARIABLES *****


----- channel: 5 unique -> 5 after strip/lower -----



,count
channel,
paid_social,2012
paid_search,2002
affiliate,1004
email,1004
organic_search,1004



----- campaign_type: 7 unique -> 7 after strip/lower -----



,count
campaign_type,
prospecting,1006
retargeting,1006
commission,1004
newsletter_platform,1004
seo_retainer,1004
brand,1001
non_brand,1001



***** DATE RANGE *****



,min,max
spend_date,2024-01-01,2026-09-30


## My questions
1. Are all 'spend_date' values within the data period stated in the data dictionary?
2. Are 'channel' and 'campaign_type' values written consistently (no different spellings of the same value)?
3. Is 'platform_conversions' empty only for 'organic_search' rows?
4. Does every 'spend_date' have exactly one row for each 'channel' and 'campaign_type' combination (no missing and no duplicate rows)?
5. Are there any rows where 'clicks' are higher than 'impressions'?
6. Does affiliate 'spend_eur' appear only some time after the first affiliate clicks?
7. Is the daily 'spend_eur' for email and organic_search the same on every day within a month?

## Next steps
- Run profile() for products (then other tables)
- Write answers under "My questions"
- Add findings to docs/data_issues.md